In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [22]:
import gym
from gym import spaces

In [23]:
import os
import random

In [24]:
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = [10, 6]

print("✅ Setup Complete! All packages for the Reinforcement Learning Framework are loaded.")

✅ Setup Complete! All packages for the Reinforcement Learning Framework are loaded.


In [25]:
class SriLankaRainfallEnv(gym.Env):
    """A custom OpenAI Gym environment using Daily and Cumulative Rainfall indices"""
    
    def __init__(self):
        super(SriLankaRainfallEnv, self).__init__()
        
        
        self.action_space = spaces.Discrete(3)
        
        
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0]), 
            high=np.array([600.0, 1000.0]), 
            dtype=np.float32
        )
        self.reset()

    def reset(self):
        """Resets the environment simulating a fresh sequence of meteorological logs"""
        rainfall_today = random.choice([random.uniform(0, 30), random.uniform(30, 150), random.uniform(150, 550)])
        cumulative_7day = rainfall_today + random.uniform(10, 450)
        
        self.state = np.array([rainfall_today, cumulative_7day], dtype=np.float32)
        return self.state

    def step(self, action):
        """The agent makes a risk determination and receives operational evaluation rewards"""
        rainfall_today, cumulative_7day = self.state
        
        
        if rainfall_today > 200 or cumulative_7day > 450:
            true_risk = 2  # Evacuate
        elif rainfall_today > 60 or cumulative_7day > 180:
            true_risk = 1  # Advisory
        else:
            true_risk = 0  # Safe
            
        
        if action == true_risk:
            reward = 10.0
        else:
            if true_risk == 2 and action < 2:
                reward = -50.0  
            else:
                reward = -10.0  
                
        done = True 
        info = {"true_risk": true_risk}
        
        return self.state, reward, done, info

print("📌 Meteorology Rainfall-Only Environment Class successfully compiled!")   

📌 Meteorology Rainfall-Only Environment Class successfully compiled!


In [ ]:
class SriLankaRealDataEnv(gym.Env):
    """An RL environment that reads step-by-step from the historical dataset"""
    def __init__(self, dataframe):
        super(SriLankaRealDataEnv, self).__init__()
        
        
        self.df = dataframe.reset_index(drop=True)
        self.total_records = len(self.df)
        self.current_index = 0
        
        
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0]), 
            high=np.array([600.0, 1000.0]), 
            dtype=np.float32
        )

    def reset(self):
        """Resets training back to the first day of historical data"""
        self.current_index = 0
        row = self.df.iloc[self.current_index]
        
        # Observation vector: [Rainfall_Today_mm, Cumulative_7Day_mm]
        self.state = np.array([row['Rainfall_Today_mm'], row['Cumulative_7Day_mm']], dtype=np.float32)
        return self.state

    def step(self, action):
        """Processes the agent's risk warning against actual recorded data"""
        row = self.df.iloc[self.current_index]
        
        
        if row['Disaster_Occurred'] == 1 or row['Rainfall_Today_mm'] > 200:
            true_risk = 2  # Evacuate
        elif row['Rainfall_Today_mm'] > 60 or row['Cumulative_7Day_mm'] > 180:
            true_risk = 1  # Advisory
        else:
            true_risk = 0  # Safe

        
        if action == true_risk:
            reward = 10.0
        else:
            if true_risk == 2 and action < 2:
                reward = -50.0  # Heavy penalty for missing a real disaster
            else:
                reward = -10.0

        
        self.current_index += 1
        done = self.current_index >= self.total_records
        
        if not done:
            next_row = self.df.iloc[self.current_index]
            self.state = np.array([next_row['Rainfall_Today_mm'], next_row['Cumulative_7Day_mm']], dtype=np.float32)
        else:
            self.state = np.array([0.0, 0.0], dtype=np.float32)
            
        return self.state, reward, done, {"true_risk": true_risk}


try:
    historical_df = pd.read_csv('srilanka_disaster_data.csv')
    print(f"Loaded {len(historical_df)} meteorological data entries for training.")
except Exception as e:
    print(f"Error loading file. Make sure the dataset exists. Details: {e}")

# 2. Instantiate environment with your data
real_env = SriLankaRealDataEnv(historical_df)


def get_discrete_state(state):
    rain_today_bin = min(int(state[0] / 20), 29) 
    rain_week_bin = min(int(state[1] / 40), 24)
    return (rain_today_bin, rain_week_bin)


q_table = np.zeros((30, 25, real_env.action_space.n))
LEARNING_RATE = 0.1
DISCOUNT_FACTOR = 0.95
epsilon = 1.0
EPSILON_DECAY = 0.995


reward_history = []

print("Commencing agent training loops across recorded dates...")

# Train using epochs/episodes over rows
for episode in range(100):  
    state = get_discrete_state(real_env.reset())
    done = False
    
    while not done:
        
        if random.uniform(0, 1) < epsilon:
            action = real_env.action_space.sample()
        else:
            action = np.argmax(q_table[state])
            
        
        next_continuous_state, reward, done, info = real_env.step(action)
        next_state = get_discrete_state(next_continuous_state)
        
        
        old_value = q_table[state][action]
        next_max = np.max(q_table[next_state])
        new_value = (1 - LEARNING_RATE) * old_value + LEARNING_RATE * (reward + DISCOUNT_FACTOR * next_max)
        q_table[state][action] = new_value
        
        state = next_state
        reward_history.append(reward)
        
    epsilon = max(0.01, epsilon * EPSILON_DECAY)

print("🏆 Model training complete! Matrix holds risk analysis weights.")

Loaded 1500 meteorological data entries for training.
Commencing agent training loops across recorded dates...


In [29]:
def predict_disaster_risk(rainfall_today, cumulative_7day):
    """
    Uses the trained Q-table matrix to predict localized risk level
    """
    
    rain_today_bin = min(int(rainfall_today / 20), 29) 
    rain_week_bin = min(int(cumulative_7day / 40), 24)
    state_bin = (rain_today_bin, rain_week_bin)
    
    
    optimal_action = np.argmax(q_table[state_bin])
    
    
    risk_labels = {
        0: "LOW RISK (Clear / Safe Conditions)",
        1: "MEDIUM RISK (Met Dept. Advisory - Stay Alert)",
        2: "HIGH RISK (Critical Threat - Immediate Evacuation Warning!)"
    }
    
    print(f"---  METEOROLOGICAL PREDICTION RESULT ---")
    print(f" Input Conditions:")
    print(f"   - Daily Rainfall Today: {rainfall_today} mm")
    print(f"   - Cumulative 7-Day Rain: {cumulative_7day} mm")
    print(f" Agent Evaluation Decision:")
    print(f" {risk_labels[optimal_action]}\n")
    
    return optimal_action

print("Prediction engine successfully operational! Ready for test scenarios.")

Prediction engine successfully operational! Ready for test scenarios.


In [30]:
print("🔬 TEST CASE 1: Standard Rain Event")
predict_disaster_risk(rainfall_today=20.0, cumulative_7day=45.0)

print("🔬 TEST CASE 2: Extreme Cyclone Simulation")
predict_disaster_risk(rainfall_today=520.0, cumulative_7day=780.0)

🔬 TEST CASE 1: Standard Rain Event
---  METEOROLOGICAL PREDICTION RESULT ---
 Input Conditions:
   - Daily Rainfall Today: 20.0 mm
   - Cumulative 7-Day Rain: 45.0 mm
 Agent Evaluation Decision:
 LOW RISK (Clear / Safe Conditions)

🔬 TEST CASE 2: Extreme Cyclone Simulation
---  METEOROLOGICAL PREDICTION RESULT ---
 Input Conditions:
   - Daily Rainfall Today: 520.0 mm
   - Cumulative 7-Day Rain: 780.0 mm
 Agent Evaluation Decision:
 LOW RISK (Clear / Safe Conditions)



np.int64(0)

In [35]:
!pip install flask flask-cors

 * Running on http://127.0.0.1:5000
Press CTRL+C to quit



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
# ==============================================================================
# PHASE 5: ADVANCED MULTI-HAZARD DISASTER AI WEB ENGINE
# ==============================================================================
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import numpy as np

app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST'])
def web_prediction_gateway():
    data = request.json
    location = data.get('location', 'Unknown Region')
    
    # Force data types to floats to prevent calculation fallback errors
    rainfall_today = float(data.get('rainfall_today', 0))
    cumulative_7day = float(data.get('cumulative_7day', 0))
    
    # 1. Map inputs to Q-table coordinates safely
    rain_today_bin = min(int(rainfall_today / 20), 29) 
    rain_week_bin = min(int(cumulative_7day / 40), 24)
    state_bin = (rain_today_bin, rain_week_bin)
    
    # 2. Extract agent risk priority
    optimal_action = int(np.argmax(q_table[state_bin]))
    
    # 3. ADVANCED SRI LANKA DISASTER CRITERIA METRICS
    # We evaluate the hazard signatures independently for distinct interface rendering
    is_flood_vulnerable = rainfall_today > 80.0
    is_landslide_vulnerable = cumulative_7day > 200.0 or (rainfall_today > 100.0 and cumulative_7day > 150.0)
    
    hazard_type = "Clear"
    display_status = 0 # 0=Safe, 1=Advisory, 2=Disaster Triggered
    
    if is_flood_vulnerable and is_landslide_vulnerable:
        hazard_type = "CRITICAL COMBINED RISK: FLOODING & LANDSLIDE SIMULTANEOUSLY!"
        display_status = 2
    elif is_landslide_vulnerable:
        hazard_type = "LANDSLIDE RISK: Soil saturation thresholds breached. High probability of structural slope failure!"
        display_status = 2
    elif is_flood_vulnerable:
        hazard_type = "FLOOD RISK: Extreme immediate surface run-off. Rivers and drainage channels overflowing!"
        display_status = 2
    elif rainfall_today > 30.0 or cumulative_7day > 80.0:
        hazard_type = "MODERATE MET ADVISORY: Weather patterns unstable. Monitor local channels."
        display_status = 1
    else:
        hazard_type = "SAFE CONDITIONS: Weather indices remain within standard bounds."
        display_status = 0

    return jsonify({
        "location": location,
        "rainfall_mm": rainfall_today,
        "cumulative_mm": cumulative_7day,
        "status_code": display_status,
        "hazard_report": hazard_type
    })

# Stop any previous threads and run cleanly
def run_server():
    app.run(port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_server).start()
print("Advanced Multi-Hazard AI Server Active on Port 5000!")

Advanced Multi-Hazard AI Server Active on Port 5000!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [20/May/2026 22:54:52] "OPTIONS /predict HTTP/1.1" 200 -
127.0.0.1 - - [20/May/2026 22:54:52] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [20/May/2026 22:55:09] "OPTIONS /predict HTTP/1.1" 200 -
127.0.0.1 - - [20/May/2026 22:55:09] "POST /predict HTTP/1.1" 200 -
